# 確率の公理と性質 — インタラクティブデモ（R版）

このノートブックでは、**コルモゴロフの確率の公理**とそこから導かれる重要な性質を、
シミュレーションと可視化を通じて体験的に学びます。

---
## 目次
1. ライブラリのインポート
2. 確率の公理（コルモゴロフ）
3. 公理1：非負性のデモ
4. 公理2：全確率 = 1 のデモ
5. 公理3：加法性のデモ（排反事象）
6. 導出される性質のデモ
   - 補事象の確率
   - 包除原理
   - 単調性
7. 大数の法則との接続（おまけ）

## 1. ライブラリのインポート

> Google Colab の R カーネルでは `install.packages()` でパッケージをインストールできます。

In [ ]:
# 必要なパッケージのインストール（未インストールの場合のみ）
pkgs <- c('ggplot2', 'dplyr', 'tidyr', 'patchwork', 'ggvenn', 'scales')
new_pkgs <- pkgs[!pkgs %in% installed.packages()[,'Package']]
if (length(new_pkgs) > 0) install.packages(new_pkgs, quiet = TRUE)

library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)
library(ggvenn)
library(scales)

set.seed(42)

# 日本語フォント設定（Google Colab Linux環境）
# showtext + IPAexフォント（Colab に同梱済み）で日本語を表示
tryCatch({
  if (!'showtext' %in% installed.packages()[,'Package'])
    install.packages('showtext', quiet = TRUE)
  library(showtext)
  # Google Fonts から Noto Sans JP を取得
  font_add_google('Noto Sans JP', 'notosans')
  showtext_auto()          # 全グラフィクスデバイスに自動適用
  showtext_opts(dpi = 120) # Colab の表示解像度に合わせる
  BASE_FAMILY <- 'notosans'
  message('showtext: Noto Sans JP を読み込みました')
}, error = function(e) {
  BASE_FAMILY <<- ''
  message('フォント設定をスキップしました: ', conditionMessage(e))
})

# 共通テーマ設定
theme_prob <- theme_bw(base_size = 13) +
  theme(
    text         = element_text(family = BASE_FAMILY),
    plot.title   = element_text(face = 'bold', size = 13, family = BASE_FAMILY),
    axis.title   = element_text(size = 11, family = BASE_FAMILY),
    legend.position = 'top'
  )

cat('ライブラリの読み込み完了\n')

---
## 2. 確率の公理（コルモゴロフの公理）

標本空間 $\Omega$ 上の確率測度 $P$ は以下の3つの公理を満たす：

| 公理 | 内容 | 数式 |
|------|------|------|
| **公理1** | 非負性 | $P(A) \geq 0 \quad \forall A \subseteq \Omega$ |
| **公理2** | 全確率 | $P(\Omega) = 1$ |
| **公理3** | 加法性 | $A \cap B = \emptyset \Rightarrow P(A \cup B) = P(A) + P(B)$ |

> これら3公理のみから、確率論の全ての性質が導出できます。

---
## 3. 公理1：非負性のデモ

サイコロ投げを例に、各事象の相対頻度（経験確率）が常に $\geq 0$ であることを確認します。

In [ ]:
n <- 10000
rolls <- sample(1:6, n, replace = TRUE)

df_freq <- data.frame(face = 1:6) |>
  mutate(freq = sapply(face, function(f) mean(rolls == f)))

p1 <- ggplot(df_freq, aes(x = factor(face), y = freq)) +
  geom_col(fill = 'steelblue', color = 'white', width = 0.7) +
  geom_hline(yintercept = 1/6, color = '#DC143C', linetype = 'dashed',
             linewidth = 1) +
  geom_text(aes(label = sprintf('%.4f', freq)),
            vjust = -0.5, size = 3.5) +
  annotate('text', x = 6.4, y = 1/6 + 0.005, label = '理論値\n1/6', 
           color = '#DC143C', size = 3.5, hjust = 1) +
  scale_y_continuous(limits = c(0, 0.25), labels = percent_format()) +
  labs(
    title = sprintf('公理1：非負性  -  サイコロ %s 回のシミュレーション', format(n, big.mark=',')),
    x = '出た目',
    y = '相対頻度（経験確率）'
  ) +
  theme_prob

print(p1)

cat('各事象の経験確率（すべて >= 0）:\n')
for (i in 1:6) {
  p <- mean(rolls == i)
  cat(sprintf('  P(目=%d) = %.4f  >= 0 : %s\n', i, p, p >= 0))
}

---
## 4. 公理2：全確率 = 1 のデモ

標本空間全体の確率 $P(\Omega)$ を累積して 1 に収束していく様子を観察します。

In [ ]:
n_sim <- 5000
rolls2 <- sample(1:6, n_sim, replace = TRUE)

ns <- seq(100, n_sim, by = 100)
totals <- sapply(ns, function(k) {
  subset <- rolls2[1:k]
  sum(sapply(1:6, function(f) mean(subset == f)))
})

df_conv <- data.frame(n = ns, total = totals)

# 左：全確率の収束グラフ
p_left <- ggplot(df_conv, aes(x = n, y = total)) +
  geom_line(color = 'steelblue', linewidth = 1) +
  geom_hline(yintercept = 1.0, color = '#DC143C', linetype = 'dashed', linewidth = 1) +
  annotate('text', x = max(ns)*0.7, y = 1.025, label = 'P(Omega) = 1',
           color = '#DC143C', size = 4) +
  scale_y_continuous(limits = c(0.95, 1.05)) +
  labs(
    title = '公理2：全確率 P(Omega) = 1 への収束',
    x = '試行回数', y = 'Sum P(目=k) の累積'
  ) +
  theme_prob

# 右：円グラフ
freq_final <- sapply(1:6, function(f) mean(rolls2 == f))
df_pie <- data.frame(
  face  = factor(1:6),
  freq  = freq_final,
  label = sprintf('目=%d\n(%.3f)', 1:6, freq_final)
)

p_right <- ggplot(df_pie, aes(x = '', y = freq, fill = face)) +
  geom_col(width = 1, color = 'white') +
  coord_polar(theta = 'y') +
  geom_text(aes(label = label),
            position = position_stack(vjust = 0.5), size = 3.2) +
  scale_fill_brewer(palette = 'Set2', guide = 'none') +
  labs(title = sprintf('標本空間の分割（合計 = %.4f）', sum(freq_final)),
       x = NULL, y = NULL) +
  theme_void(base_size = 13) +
  theme(plot.title = element_text(face = 'bold', size = 12, hjust = 0.5))

print(p_left + p_right)
cat(sprintf('全確率の合計 = %.6f  (理論値 = 1.000000)\n', sum(freq_final)))

---
## 5. 公理3：加法性のデモ（排反事象）

$A \cap B = \emptyset$ のとき、$P(A \cup B) = P(A) + P(B)$

サイコロの例：$A = \{1, 2\}$、$B = \{5, 6\}$ は排反事象。

In [ ]:
n <- 20000
rolls3 <- sample(1:6, n, replace = TRUE)

A <- c(1, 2)
B <- c(5, 6)

pA    <- mean(rolls3 %in% A)
pB    <- mean(rolls3 %in% B)
pAuB  <- mean(rolls3 %in% union(A, B))
pAuB_theory <- pA + pB

# 左：棒グラフ
df_bar <- data.frame(
  label = factor(c('P(A)', 'P(B)', 'P(A∪B)\n実測', 'P(A)+P(B)\n加法性'),
                 levels = c('P(A)', 'P(B)', 'P(A∪B)\n実測', 'P(A)+P(B)\n加法性')),
  value = c(pA, pB, pAuB, pAuB_theory),
  color_group = c('A', 'B', 'AuB', 'theory')
)

p_bar <- ggplot(df_bar, aes(x = label, y = value, fill = color_group)) +
  geom_col(color = 'white', width = 0.6) +
  geom_text(aes(label = sprintf('%.4f', value)), vjust = -0.4, size = 3.5) +
  scale_fill_manual(values = c('A'='#4C72B0','B'='#DD8452','AuB'='#55A868','theory'='#C44E52'),
                    guide = 'none') +
  scale_y_continuous(limits = c(0, 0.8)) +
  labs(title = '公理3：加法性の確認\nA={1,2}, B={5,6}（排反）',
       x = NULL, y = '確率') +
  theme_prob

# 右：ベン図
venn_data <- list('A = {1,2}' = which(rolls3 %in% A),
                  'B = {5,6}' = which(rolls3 %in% B))

p_venn <- ggvenn(venn_data,
                 fill_color = c('#4C72B0', '#DD8452'),
                 fill_alpha = 0.5,
                 stroke_size = 0.8,
                 text_size = 4,
                 set_name_size = 4) +
  labs(title = '排反事象のベン図\n（A∩B = 空集合）') +
  theme(plot.title = element_text(face = 'bold', size = 12, hjust = 0.5))

print(p_bar + p_venn)

cat(sprintf('P(A)           = %.4f  （理論値 2/6 = %.4f）\n', pA, 2/6))
cat(sprintf('P(B)           = %.4f  （理論値 2/6 = %.4f）\n', pB, 2/6))
cat(sprintf('P(A∪B) 実測    = %.4f  （理論値 4/6 = %.4f）\n', pAuB, 4/6))
cat(sprintf('P(A)+P(B)      = %.4f\n', pAuB_theory))
cat(sprintf('差（|実測 - 加法性|） = %.6f\n', abs(pAuB - pAuB_theory)))

---
## 6. 導出される性質のデモ

3公理から**定理として**導かれる重要な性質を確認します。

### 6-1. 補事象の確率：$P(A^c) = 1 - P(A)$

In [ ]:
n <- 10000
rolls4 <- sample(1:6, n, replace = TRUE)

events_comp <- list(
  list(name = '偶数 {2,4,6}', cond = function(x) x %% 2 == 0),
  list(name = '3以下 {1,2,3}', cond = function(x) x <= 3),
  list(name = '6の目 {6}',     cond = function(x) x == 6)
)

plots_comp <- lapply(events_comp, function(ev) {
  pA  <- mean(ev$cond(rolls4))
  pAc <- mean(!ev$cond(rolls4))
  tot <- pA + pAc
  df <- data.frame(
    label = factor(c('P(A)', 'P(Ac)', 'P(A)+P(Ac)'),
                   levels = c('P(A)', 'P(Ac)', 'P(A)+P(Ac)')),
    value = c(pA, pAc, tot)
  )
  ggplot(df, aes(x = label, y = value, fill = label)) +
    geom_col(color = 'white', width = 0.6) +
    geom_hline(yintercept = 1.0, color = '#DC143C', linetype = 'dashed') +
    geom_text(aes(label = sprintf('%.3f', value)), vjust = -0.4, size = 3.2) +
    scale_fill_manual(values = c('P(A)'='#4C72B0','P(Ac)'='#DD8452','P(A)+P(Ac)'='#55A868'),
                      guide = 'none') +
    scale_y_continuous(limits = c(0, 1.25)) +
    labs(title = sprintf('A = %s\nP(A)=%.3f, P(Ac)=%.3f', ev$name, pA, pAc),
         x = NULL, y = '確率') +
    theme_prob +
    theme(plot.title = element_text(size = 10))
})

print(wrap_plots(plots_comp, ncol = 3) +
  plot_annotation(title = '補事象の確率：P(Ac) = 1 - P(A)',
                  theme = theme(plot.title = element_text(face='bold', size=13))))

### 6-2. 包除原理：$P(A \cup B) = P(A) + P(B) - P(A \cap B)$

一般の（排反でない）事象に対する加法の公式です。

In [ ]:
n <- 30000
rolls5 <- sample(1:6, n, replace = TRUE)

# A = 偶数 {2,4,6}、B = 4以上 {4,5,6}  → 重なり = {4,6}
A_mask <- rolls5 %% 2 == 0
B_mask <- rolls5 >= 4

pA    <- mean(A_mask)
pB    <- mean(B_mask)
pAB   <- mean(A_mask & B_mask)
pAuB  <- mean(A_mask | B_mask)
pAuB_ie <- pA + pB - pAB

# 左：棒グラフ
df_ie <- data.frame(
  label = factor(
    c('P(A)', 'P(B)', 'P(A∩B)', 'P(A∪B)\n実測', 'P(A)+P(B)\n-P(A∩B)'),
    levels = c('P(A)', 'P(B)', 'P(A∩B)', 'P(A∪B)\n実測', 'P(A)+P(B)\n-P(A∩B)')
  ),
  value = c(pA, pB, pAB, pAuB, pAuB_ie),
  grp   = c('a','b','c','d','e')
)

p_ie_bar <- ggplot(df_ie, aes(x = label, y = value, fill = grp)) +
  geom_col(color = 'white', width = 0.65) +
  geom_text(aes(label = sprintf('%.4f', value)), vjust = -0.4, size = 3.2) +
  scale_fill_manual(
    values = c('a'='#4C72B0','b'='#DD8452','c'='#C44E52','d'='#55A868','e'='#8172B2'),
    guide = 'none') +
  scale_y_continuous(limits = c(0, 1.0)) +
  labs(title = '包除原理の確認\nA=偶数, B=4以上', x = NULL, y = '確率') +
  theme_prob

# 右：ベン図
venn_data2 <- list(
  'A=偶数\n{2,4,6}'  = which(A_mask),
  'B=4以上\n{4,5,6}' = which(B_mask)
)

p_ie_venn <- ggvenn(venn_data2,
                    fill_color = c('#4C72B0', '#DD8452'),
                    fill_alpha = 0.5,
                    stroke_size = 0.8,
                    text_size = 4,
                    set_name_size = 3.5) +
  labs(title = 'ベン図（A∩B = {4,6}）') +
  theme(plot.title = element_text(face = 'bold', size = 12, hjust = 0.5))

print(p_ie_bar + p_ie_venn)

cat(sprintf('P(A)              = %.4f  （理論値 %.4f）\n', pA,    3/6))
cat(sprintf('P(B)              = %.4f  （理論値 %.4f）\n', pB,    3/6))
cat(sprintf('P(A∩B)            = %.4f  （理論値 %.4f）\n', pAB,   2/6))
cat(sprintf('P(A∪B) 実測       = %.4f  （理論値 %.4f）\n', pAuB,  4/6))
cat(sprintf('P(A)+P(B)-P(A∩B) = %.4f\n', pAuB_ie))
cat(sprintf('差                 = %.6f\n', abs(pAuB - pAuB_ie)))

### 6-3. 単調性：$A \subseteq B \Rightarrow P(A) \leq P(B)$

In [ ]:
n <- 20000
rolls6 <- sample(1:6, n, replace = TRUE)

nested <- list(
  list(name = '{1}',       cond = function(x) x == 1),
  list(name = '{1,2}',     cond = function(x) x <= 2),
  list(name = '{1,2,3}',   cond = function(x) x <= 3),
  list(name = '{1,2,3,4}', cond = function(x) x <= 4)
)

df_mono <- data.frame(
  label  = sapply(nested, function(e) e$name),
  sim    = sapply(nested, function(e) mean(e$cond(rolls6))),
  theory = c(1/6, 2/6, 3/6, 4/6)
) |>
  mutate(label = factor(label, levels = label)) |>
  pivot_longer(cols = c(sim, theory), names_to = 'type', values_to = 'prob') |>
  mutate(type = recode(type, sim = 'シミュレーション', theory = '理論値'))

p_mono <- ggplot(df_mono, aes(x = label, y = prob, fill = type, group = type)) +
  geom_col(position = position_dodge(width = 0.65), width = 0.6, color = 'white') +
  geom_line(aes(color = type), position = position_dodge(width = 0.65),
            linewidth = 0.8) +
  geom_point(aes(color = type), position = position_dodge(width = 0.65),
             size = 3, shape = 21, fill = 'white', stroke = 1.5) +
  scale_fill_manual(values  = c('シミュレーション'='steelblue', '理論値'='coral')) +
  scale_color_manual(values = c('シミュレーション'='steelblue', '理論値'='coral')) +
  scale_y_continuous(limits = c(0, 0.8)) +
  labs(
    title = '単調性：A1 ⊂ A2 ⊂ A3 ⊂ A4  =>  P(A1) <= P(A2) <= P(A3) <= P(A4)',
    x = '事象', y = '確率', fill = NULL, color = NULL
  ) +
  theme_prob

print(p_mono)

cat('事象の包含関係と確率の単調性：\n')
probs_vec <- sapply(nested, function(e) mean(e$cond(rolls6)))
for (i in seq_along(nested)) {
  cat(sprintf('  P(%-12s) = %.4f  （理論値 %.4f）\n',
              nested[[i]]$name, probs_vec[i], i/6))
}
cat(sprintf('\n単調増加を確認: %s\n', all(diff(probs_vec) >= 0)))

---
## 7. おまけ：大数の法則との接続

試行回数を増やすほど経験確率が真の確率に収束していく様子を観察します。

$$\frac{1}{n}\sum_{i=1}^n \mathbf{1}_{A}(\omega_i) \xrightarrow{n\to\infty} P(A)$$

In [ ]:
set.seed(0)
N <- 50000
rolls_large <- sample(1:6, N, replace = TRUE)

n_trials <- unique(round(10^seq(1, log10(N), length.out = 300)))

events_lln <- list(
  list(label='P(目=1) = 1/6',  mask=rolls_large==1,        theory=1/6, color='#4C72B0'),
  list(label='P(偶数) = 1/2',  mask=rolls_large%%2==0,     theory=1/2, color='#DD8452'),
  list(label='P(4以上) = 1/2', mask=rolls_large>=4,        theory=1/2, color='#55A868')
)

plots_lln <- lapply(events_lln, function(ev) {
  freq_seq <- sapply(n_trials, function(k) mean(ev$mask[1:k]))
  df <- data.frame(n = n_trials, freq = freq_seq)
  ggplot(df, aes(x = n, y = freq)) +
    geom_ribbon(aes(ymin = ev$theory - 0.02, ymax = ev$theory + 0.02),
                fill = '#DC143C', alpha = 0.08) +
    geom_line(color = ev$color, linewidth = 0.9, alpha = 0.85) +
    geom_hline(yintercept = ev$theory, color = '#DC143C',
               linetype = 'dashed', linewidth = 1) +
    annotate('text', x = max(n_trials)*0.3,
             y = ev$theory + 0.035,
             label = sprintf('理論値 = %.4f', ev$theory),
             color = '#DC143C', size = 3.5) +
    scale_x_log10(labels = label_comma()) +
    labs(title = ev$label,
         x = '試行回数 n（対数スケール）',
         y = '経験確率') +
    theme_prob +
    theme(plot.title = element_text(size = 11))
})

print(wrap_plots(plots_lln, ncol = 3) +
  plot_annotation(title = '大数の法則：経験確率の真の確率への収束',
                  theme = theme(plot.title = element_text(face='bold', size=13))))

cat('最終的な経験確率（n = 50,000）：\n')
for (ev in events_lln) {
  emp <- mean(ev$mask)
  cat(sprintf('  %-20s 経験値=%.5f  誤差=%.5f\n', ev$label, emp, abs(emp - ev$theory)))
}

---
## まとめ

| 性質 | 数式 | 確認 |
|------|------|------|
| 非負性（公理1） | $P(A) \geq 0$ | ✓ セル3 |
| 全確率（公理2） | $P(\Omega) = 1$ | ✓ セル4 |
| 加法性（公理3） | $P(A \cup B) = P(A)+P(B)$ （排反） | ✓ セル5 |
| 補事象 | $P(A^c) = 1 - P(A)$ | ✓ セル6-1 |
| 包除原理 | $P(A \cup B) = P(A)+P(B)-P(A \cap B)$ | ✓ セル6-2 |
| 単調性 | $A \subseteq B \Rightarrow P(A) \leq P(B)$ | ✓ セル6-3 |
| 大数の法則 | 経験確率 → 真の確率 | ✓ セル7 |

> **ポイント**：コルモゴロフの3公理は「最小限の仮定」であり、  
> これだけから確率論の全ての豊かな性質が論理的に導出されます。